# Contoso Private Banking MCP server

A 6-tool, intent-level MCP server for the morning workflow of a Swiss private-banking
relationship manager. The headline tool, `cpb_prepare_client_briefing`, returns a
fully-stitched briefing for one client meeting - portfolio summary, IPS drift,
recent activity, CRM flags, relevant research, and next-best-actions - in **one** call.

This lab is the **intent-level counterpoint** to [`08-05-contoso-pmo-mcp/`](../08-05-contoso-pmo-mcp/),
which exposes 37 endpoint-style CRUD tools for the same Azure plumbing. The
pedagogical contrast is the lesson - see [`08-05b-00-contoso-private-banking-mcp.md`](08-05b-00-contoso-private-banking-mcp.md)
for the design rationale and the [Anthropic guidance](https://www.anthropic.com/engineering/writing-tools-for-agents)
this lab implements.

This notebook is the **canonical creator** of the `aria-rm-briefing-agent`. Re-runs
return the existing agent rather than creating a duplicate.

```
assets/contoso-private-banking-dataset/   ←── synthetic Contoso Private Investments KB
    │
    ▼
private-banking-mcp/                       ←── Azure Functions app
  function_app.py            6 mcpToolTrigger wrappers
  kb.py                      Intent-level business logic + drift math + citations
  host.json                  Standard GA extension bundle [4, 5)
    │
    ▼ SSE endpoint /runtime/webhooks/mcp/sse
    │
PromptAgentDefinition       ←── tools=[{type:'mcp', require_approval:'never'}]
    │
    ▼ project_client.agents.create_version on project-admin-{suffix}
Foundry Agent (versioned)  ←── aria-rm-briefing-agent v1, v2, …
```


## Prerequisites

1. **Hub deployment complete** - the admin project (`project-admin-{suffix}` on `aif-core-{suffix}`) must already exist. Endpoint is derived deterministically from the subscription ID.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - `az login` with `Contributor` + `User Access Administrator` on the target subscription, plus `Azure AI Developer` on the admin project.
4. **Required RBAC roles on the function-app storage** (assigned automatically by this notebook):
   - `Storage Blob Data Owner` - deployment package access
   - `Storage Queue Data Contributor` - MCP SSE transport (queue creation)
   - `Storage Table Data Contributor` - host metadata
5. **`.env`** - only `CHAT_MODEL` is required (e.g. `CHAT_MODEL=gpt-4.1-mini`).

Optional `.env` overrides:
- `PRIVATE_BANKING_MCP_RESOURCE_GROUP` (default: `rg-foundry-private-banking-mcp`)
- `PRIVATE_BANKING_MCP_LOCATION` (default: `swedencentral`)
- `PRIVATE_BANKING_FUNC_APP_NAME` (default: derived as `func-private-banking-mcp-{md5(sub-id+rg)[:6]}`)


## Imports and configuration

In [1]:
import hashlib
import json
import os
import subprocess
import time
import zipfile
from pathlib import Path

from dotenv import load_dotenv

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

SUBSCRIPTION_ID = (
    os.environ.get('AZURE_SUBSCRIPTION_ID')
    or subprocess.run('az account show --query id -o tsv',
                      shell=True, capture_output=True, text=True).stdout.strip()
)
SUFFIX           = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

PB_MCP_RG  = os.environ.get('PRIVATE_BANKING_MCP_RESOURCE_GROUP', 'rg-foundry-private-banking-mcp')
PB_MCP_LOC = os.environ.get('PRIVATE_BANKING_MCP_LOCATION', 'swedencentral')
_mcp_suffix    = hashlib.md5(f'{SUBSCRIPTION_ID}-{PB_MCP_RG}'.encode()).hexdigest()[:6]
STORAGE_NAME   = f'stprivatebankmcp{_mcp_suffix}'
FUNC_APP_NAME  = os.environ.get('PRIVATE_BANKING_FUNC_APP_NAME') or f'func-private-banking-mcp-{_mcp_suffix}'

AGENT_NAME = 'aria-rm-briefing-agent'

SOURCE_DIR = repo_root / '08-agents' / '08-05b-contoso-private-banking-mcp' / 'private-banking-mcp'
ZIP_PATH   = repo_root / '08-agents' / '08-05b-contoso-private-banking-mcp' / 'private-banking-mcp.zip'

print(f'Subscription     : {SUBSCRIPTION_ID}')
print(f'Admin endpoint   : {PROJECT_ENDPOINT}')
print(f'Resource group   : {PB_MCP_RG}')
print(f'Location         : {PB_MCP_LOC}')
print(f'Storage account  : {STORAGE_NAME}')
print(f'Function app     : {FUNC_APP_NAME}')
print(f'Agent name       : {AGENT_NAME}')
print(f'Chat model       : {CHAT_MODEL}')
print(f'Source dir       : {SOURCE_DIR}')


Subscription     : 00000000-0000-0000-0000-000000000000
Admin endpoint   : https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
Resource group   : rg-foundry-private-banking-mcp
Location         : swedencentral
Storage account  : stprivatebankmcp97aab2
Function app     : func-private-banking-mcp-97aab2
Agent name       : aria-rm-briefing-agent
Chat model       : gpt-4.1-mini
Source dir       : <repo-root>/08-agents/08-05b-contoso-private-banking-mcp/private-banking-mcp


## Authentication

In [2]:
from azure.identity import DefaultAzureCredential

foundry_credential = DefaultAzureCredential()
print('DefaultAzureCredential initialised.')


DefaultAzureCredential initialised.


---
## Phase 1: Build the MCP server

Writes `host.json` and `requirements.txt`, verifies `function_app.py` and `kb.py`
are present, and bundles the synthetic Contoso Private Investments dataset into
`private-banking-mcp/data/`.


In [3]:
import json as _json
import shutil

SOURCE_DIR.mkdir(exist_ok=True)

(SOURCE_DIR / 'host.json').write_text(_json.dumps({
    'version': '2.0',
    'extensionBundle': {
        'id': 'Microsoft.Azure.Functions.ExtensionBundle',
        'version': '[4.0.0, 5.0.0)'
    }
}, indent=2) + '\n')

(SOURCE_DIR / 'requirements.txt').write_text('azure-functions\n')

for _name in ('function_app.py', 'kb.py'):
    _p = SOURCE_DIR / _name
    assert _p.exists(), f'{_name} missing from {SOURCE_DIR}'
    print(f'  {_name}: {_p.stat().st_size:,} bytes')

_data_dest = SOURCE_DIR / 'data'
if _data_dest.exists():
    shutil.rmtree(_data_dest)
shutil.copytree(str(repo_root / 'assets' / 'contoso-private-banking-dataset'), _data_dest)

print(f'\nSource directory: {SOURCE_DIR.resolve()}')
print(f'  host.json       : ready (standard GA extension bundle)')
print(f'  requirements.txt: ready')
print(f'  data/           : {len(list(_data_dest.rglob("*.json")))} JSON files bundled')


  function_app.py: 11,098 bytes
  kb.py: 29,742 bytes

Source directory: <repo-root>/08-agents/08-05b-contoso-private-banking-mcp/private-banking-mcp
  host.json       : ready (standard GA extension bundle)
  requirements.txt: ready
  data/           : 25 JSON files bundled


---
## Phase 2: Deploy to Azure

Provisions a dedicated resource group with a Flex Consumption function app using
managed identity for storage (no shared keys). Assigns the three storage roles
required for MCP SSE transport and sets `DATA_DIR=data`.


In [4]:
def _run(cmd, label):
    """Run an az CLI command via subprocess. Prints ✓ or ✗ and returns the result."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  ✓ {label}')
    else:
        print(f'  ✗ {label}')
        err = (result.stderr or result.stdout).strip()
        if err:
            print(f'    {err[:400]}')
    return result


STORAGE_SCOPE = (
    f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{PB_MCP_RG}'
    f'/providers/Microsoft.Storage/storageAccounts/{STORAGE_NAME}'
)

print('Creating resource group...')
_run(
    f'az group create --name "{PB_MCP_RG}" --location "{PB_MCP_LOC}" -o none',
    f"resource group '{PB_MCP_RG}'")

# `--public-network-access Enabled` is required: recent Azure subscription defaults
# disable public endpoints on new storage accounts, which breaks both the
# `az storage container create --auth-mode login` call below AND
# `az functionapp create` (which refuses to bind a function app to a
# network-restricted storage account without VNet integration).
# `--allow-shared-key-access false` keeps the secure key-less posture (managed
# identity is used for AzureWebJobsStorage); only the *network* layer is opened.
print('Creating storage account...')
_run(
    f'az storage account create -g "{PB_MCP_RG}" -n "{STORAGE_NAME}" '
    f'-l "{PB_MCP_LOC}" --sku Standard_LRS '
    f'--allow-shared-key-access false --public-network-access Enabled -o none',
    f"storage account '{STORAGE_NAME}'")

print('Assigning current user role for container creation...')
_current_user = subprocess.run(
    'az ad signed-in-user show --query id -o tsv',
    shell=True, capture_output=True, text=True).stdout.strip()
_run(
    f'az role assignment create --assignee "{_current_user}" '
    f'--role "Storage Blob Data Contributor" --scope "{STORAGE_SCOPE}" -o none',
    'Storage Blob Data Contributor → current user')
print('  … waiting 15 s for RBAC to propagate...')
time.sleep(15)

_run(
    f'az storage container create --account-name "{STORAGE_NAME}" '
    f'--name deployments --auth-mode login -o none',
    'deployments blob container')

print(f"Creating function app '{FUNC_APP_NAME}' (~2 min)...")
_run(
    f'az functionapp create -g "{PB_MCP_RG}" -n "{FUNC_APP_NAME}" '
    f'--storage-account "{STORAGE_NAME}" --runtime python --runtime-version 3.11 '
    f'--flexconsumption-location "{PB_MCP_LOC}" '
    f'--deployment-storage-container-name deployments '
    f'--deployment-storage-auth-type SystemAssignedIdentity -o none',
    f"function app '{FUNC_APP_NAME}'")
print('  … waiting 30 s for managed identity to initialize...')
time.sleep(30)

print('Assigning storage roles to managed identity...')
_PRINCIPAL = subprocess.run(
    f'az functionapp identity show -g "{PB_MCP_RG}" -n "{FUNC_APP_NAME}" '
    f'--query principalId -o tsv',
    shell=True, capture_output=True, text=True).stdout.strip()

if not _PRINCIPAL:
    print('  ✗ managed identity not found - function app may not have provisioned')
else:
    print(f'  ✓ managed identity principal: {_PRINCIPAL}')
    for _role in [
        'Storage Blob Data Owner',
        'Storage Queue Data Contributor',
        'Storage Table Data Contributor',
    ]:
        _run(
            f'az role assignment create --assignee-object-id "{_PRINCIPAL}" '
            f'--assignee-principal-type ServicePrincipal '
            f'--role "{_role}" --scope "{STORAGE_SCOPE}" -o none',
            _role)

print('Configuring identity-based storage connection...')
_run(
    f'az functionapp config appsettings delete -g "{PB_MCP_RG}" '
    f'-n "{FUNC_APP_NAME}" --setting-names AzureWebJobsStorage -o none',
    'removed key-based AzureWebJobsStorage')
_run(
    f'az functionapp config appsettings set -g "{PB_MCP_RG}" '
    f'-n "{FUNC_APP_NAME}" '
    f'--settings AzureWebJobsStorage__accountName={STORAGE_NAME} '
    f'AzureWebJobsStorage__credential=managedidentity -o none',
    'set managed identity storage connection')
_run(
    f'az functionapp config appsettings set -g "{PB_MCP_RG}" '
    f'-n "{FUNC_APP_NAME}" --settings DATA_DIR=data -o none',
    'DATA_DIR=data')


Creating resource group...
  ✓ resource group 'rg-foundry-private-banking-mcp'
Creating storage account...
  ✓ storage account 'stprivatebankmcp97aab2'
Assigning current user role for container creation...
  ✓ Storage Blob Data Contributor → current user
  … waiting 15 s for RBAC to propagate...
  ✓ deployments blob container
Creating function app 'func-private-banking-mcp-97aab2' (~2 min)...
  ✗ function app 'func-private-banking-mcp-97aab2'
    ERROR: Cannot change the site func-private-banking-mcp-97aab2 to the App Service Plan ASP-rgfoundryprivatebankingmcp-c0e3 due to hosting constraints.
  … waiting 30 s for managed identity to initialize...
Assigning storage roles to managed identity...
  ✓ managed identity principal: 67bc2a77-6ab1-4d78-b03a-16ccc0529bdd
  ✓ Storage Blob Data Owner
  ✓ Storage Queue Data Contributor
  ✓ Storage Table Data Contributor
Configuring identity-based storage connection...
  ✓ removed key-based AzureWebJobsStorage
  ✓ set managed identity storage connec

CompletedProcess(args='az functionapp config appsettings set -g "rg-foundry-private-banking-mcp" -n "func-private-banking-mcp-97aab2" --settings DATA_DIR=data -o none', returncode=0, stdout='', stderr='')

---
## Phase 3: Package and deploy code

Zips `private-banking-mcp/` (including the bundled dataset) and deploys it to the
function app. Re-run this phase alone whenever `function_app.py`, `kb.py`, or the
data files change.


In [5]:
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for _fp in SOURCE_DIR.rglob('*'):
        if _fp.is_file():
            zf.write(_fp, _fp.relative_to(SOURCE_DIR))

print(f'Zip archive: {ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes)')
print('Deploying code...')
_deploy = subprocess.run(
    f'az functionapp deployment source config-zip '
    f'-g "{PB_MCP_RG}" -n "{FUNC_APP_NAME}" --src "{ZIP_PATH}"',
    shell=True, capture_output=True, text=True)
if _deploy.returncode == 0 or '202' in (_deploy.stdout + _deploy.stderr):
    print('  ✓ code deployed')
else:
    print(f'  ✗ code deployment failed')
    print(f'    {(_deploy.stdout + _deploy.stderr)[:400]}')


Zip archive: <repo-root>/08-agents/08-05b-contoso-private-banking-mcp/private-banking-mcp.zip (53,545 bytes)
Deploying code...
  ✓ code deployed


---
## Phase 4: Retrieve function URL and key

Reads the `mcp_extension` system key. This key is generated by the MCP extension
after first load - the cell retries up to 6 times (120 seconds total) while the
extension initialises.


In [6]:
FUNC_BASE_URL = f'https://{FUNC_APP_NAME}.azurewebsites.net'

print('Retrieving MCP system key (mcp_extension)...')
MCP_KEY = ''
for _attempt in range(6):
    _r = subprocess.run(
        f'az functionapp keys list -g "{PB_MCP_RG}" -n "{FUNC_APP_NAME}" '
        f'--query "systemKeys.mcp_extension" -o tsv',
        shell=True, capture_output=True, text=True)
    MCP_KEY = _r.stdout.strip()
    if MCP_KEY and MCP_KEY != 'None':
        print('  ✓ MCP system key (mcp_extension) retrieved')
        break
    print(f'  … not ready yet, waiting 20 s (attempt {_attempt + 1}/6)...')
    time.sleep(20)

if not MCP_KEY or MCP_KEY == 'None':
    print('  ✗ MCP system key not found - check function app logs')
    MCP_KEY = ''

MCP_SSE_URL = f'{FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code={MCP_KEY}'

print(f'Function base URL : {FUNC_BASE_URL}')
print(f'MCP SSE endpoint  : {FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code=<key>')


Retrieving MCP system key (mcp_extension)...
  ✓ MCP system key (mcp_extension) retrieved
Function base URL : https://func-private-banking-mcp-97aab2.azurewebsites.net
MCP SSE endpoint  : https://func-private-banking-mcp-97aab2.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>


---
## Phase 4.5: Smoke-test the MCP server

Verifies the function app is reachable and the MCP SSE endpoint accepts the
`mcp_extension` system key **before** the agent is created. Catches deployment
problems at deploy time rather than as opaque `tool_server_error / 504` failures
during agent runs.


In [7]:
import urllib.request
import urllib.error
import socket

print('Smoke-testing MCP server reachability...')

try:
    _req = urllib.request.Request(FUNC_BASE_URL, method='HEAD')
    with urllib.request.urlopen(_req, timeout=60) as _r:
        print(f'  ✓ Function app root responding (HTTP {_r.status})')
except urllib.error.HTTPError as _e:
    print(f'  ✓ Function app root responding (HTTP {_e.code})')
except (socket.timeout, urllib.error.URLError) as _e:
    raise RuntimeError(
        f"Function app '{FUNC_APP_NAME}' did not respond within 60 s.\n"
        f"  Check: az functionapp show -g {PB_MCP_RG} -n {FUNC_APP_NAME} --query state -o tsv\n"
        f"  Error: {_e}"
    ) from _e

print('  Opening MCP SSE stream...')
try:
    _req = urllib.request.Request(MCP_SSE_URL, headers={'Accept': 'text/event-stream'})
    with urllib.request.urlopen(_req, timeout=30) as _r:
        if _r.status != 200:
            raise RuntimeError(f'MCP SSE returned HTTP {_r.status}')
        _chunk = _r.read(256)
        if not _chunk:
            raise RuntimeError('MCP SSE opened but produced no data - extension may not be ready')
        print(f'  ✓ MCP SSE stream alive ({len(_chunk)} bytes received in first chunk)')
except urllib.error.HTTPError as _e:
    _body = _e.read(400) if hasattr(_e, 'read') else b''
    raise RuntimeError(
        f"MCP SSE endpoint returned HTTP {_e.code}.\n"
        f"  Body: {_body!r}"
    ) from _e
except (socket.timeout, urllib.error.URLError) as _e:
    raise RuntimeError(
        f"MCP SSE endpoint timed out or unreachable.\n"
        f"  Most common cause: function-app cold start exceeded 30 s - re-run this cell.\n"
        f"  Error: {_e}"
    ) from _e

print('\n  ✓ MCP server smoke test passed - safe to create the agent')


Smoke-testing MCP server reachability...
  ✓ Function app root responding (HTTP 200)
  Opening MCP SSE stream...
  ✓ MCP SSE stream alive (256 bytes received in first chunk)

  ✓ MCP server smoke test passed — safe to create the agent


---
## Phase 5: Create the Foundry agent on the admin project

Creates the `aria-rm-briefing-agent` on `project-admin-{suffix}` via the
versioned-agent API (`project_client.agents.create_version` +
`PromptAgentDefinition`). The MCP tool is attached as a tool-definition dict
with `require_approval='never'` baked in - no per-run `ToolSet` plumbing is
needed in [`08-05b-02`](08-05b-02-private-banking-agent-queries.ipynb).

The agent's instructions explicitly steer it toward the intent-level tools -
picking `cpb_prepare_client_briefing` for "prepare for meeting" requests rather
than composing the lower-level tools.

**Idempotency:** if a version of this agent already exists, the cell reuses the
latest version rather than appending another. Force a new version by passing
`force_new_version=True` (e.g. after editing the instructions).


In [8]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.core.exceptions import ResourceNotFoundError

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=foundry_credential)

_INSTRUCTIONS = (
    'You are Aria, a research and briefing assistant for senior investment '
    'counsellors at Contoso Private Investments (a Swiss private bank). You support '
    'the morning workflow: client meeting prep, drift analysis, recent activity, '
    'and lookups against the in-house research/commentary/regulatory corpus.\n\n'
    'TOOL SELECTION (intent-level)\n'
    'Your tools are designed for whole RM workflows, not for low-level CRUD. Pick '
    'the highest-intent tool for each request:\n'
    '  • For "prepare for the meeting with X" / "what should I know about X" / '
    '"give me a brief on X" → call cpb_prepare_client_briefing. This returns the '
    'full briefing in one call (portfolio, drift, transactions, CRM, research, '
    'next-best-actions). Do NOT compose lower-level tools to recreate it.\n'
    '  • For drift-only / rebalancing questions → cpb_analyze_portfolio_drift\n'
    '  • For recent activity / "what trades happened" → cpb_summarize_recent_activity\n'
    '  • For research / commentary / regulatory lookups → cpb_find_relevant_research\n'
    '  • For just identity + IPS basics → cpb_get_client_context\n'
    '  • For the long tail (raw collection lookups, fetching a specific document) → '
    'cpb_run_query\n\n'
    'RESPONSE FORMAT\n'
    'Use response_format="concise" (the default) for synthesis prompts. Use '
    '"detailed" only when you need IDs to chain a follow-up call.\n\n'
    'CITATIONS\n'
    'Every tool returns citations. Reproduce them in your reply (research/<id>, '
    'market_commentary/<id>, ips/<client_id>, etc.) - they are the audit handle '
    'Compliance and the RM rely on. Do not invent citations or paraphrase the '
    'corpus without one.\n\n'
    'GROUNDING\n'
    'Never guess client names, ISINs, or amounts. Always call a tool. If a client '
    'ID does not exist, the error message will name the valid IDs - try one of '
    'them rather than apologising.\n\n'
    'OUT-OF-SCOPE\n'
    'You are read-only. You do not propose, place, or settle trades. You do not '
    'give tax advice. For trade execution and tax matters, hand back to the RM.'
)

force_new_version = False  # set True to bump a new version on re-run

# Reuse-or-create: list_versions raises ResourceNotFoundError when the agent
# does not yet exist; treat that as "create the first version".
try:
    existing_versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
except ResourceNotFoundError:
    existing_versions = []

if existing_versions and not force_new_version:
    agent = existing_versions[0]
    print(f"Reusing existing agent '{agent.name}' v{agent.version}")
else:
    agent = project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=CHAT_MODEL,
            instructions=_INSTRUCTIONS,
            tools=[
                {
                    'type': 'mcp',
                    'server_label': 'contoso_private_banking',
                    'server_url': MCP_SSE_URL,
                    'require_approval': 'never',
                },
            ],
        ),
        description='Aria - Contoso Private Investments RM briefing assistant. Intent-level MCP server.',
    )
    print(f"Created agent '{agent.name}' v{agent.version}")

print(f'MCP server label : contoso_private_banking')
print(f'MCP SSE endpoint : {FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code=<key>')


Created agent 'aria-rm-briefing-agent' v1
MCP server label : contoso_private_banking
MCP SSE endpoint : https://func-private-banking-mcp-97aab2.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>
